# Basic Tutorial: Systemic Tau and Discrete Extramental Clock (RECD)

This notebook demonstrates how to use the `systemictau` package to process a multivariate time series, extract its ordinal structure, accumulate internal time (RECD), and detect ontological transitions (Capa 3).

## 1. Import Libraries and Generate Synthetic Data

We will use coupled logistic maps simulating a complex system that enters chaos and then undergoes a structural perturbation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import systemictau as st

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Logistic map generator
def logistic_map(n_steps, r, x0=0.5):
    x = np.zeros(n_steps)
    x[0] = x0
    for i in range(1, n_steps):
        x[i] = r * x[i-1] * (1 - x[i-1])
    return x

np.random.seed(42)
T_steps = 2000
N_comp = 4

X = np.zeros((T_steps, N_comp))
for i in range(N_comp):
    X[:, i] = logistic_map(T_steps, 3.8, x0=0.1 + i*0.05)

print(f"Generated data. Dimensions: {X.shape}")

## 2. Computation of Ordinal Coherence (Systemic Tau)

We compute the global and per-module $\tau_s(k)$ using a sliding window of $n=13$.

In [ ]:
taus_global, taus_per_module = st.compute_taus(X, window_size=13)

plt.plot(taus_global, color='purple', alpha=0.7)
plt.axhline(0.41, color='green', linestyle='--', label='Chaotic Threshold')
plt.axhline(-0.41, color='red', linestyle='--', label='Anti-synchronization')
plt.title("Evolution of Global Systemic Tau")
plt.legend()
plt.show()

## 3. The Discrete Extramental Clock (RECD)

We accumulate the internal time $T$ based on the Feigenbaum constant and the gate function $g(\tau_s)$.

In [ ]:
T_series, dtk_series, gate_series, depths = st.accumulate_time(taus_global)

fig, ax1 = plt.subplots()
ax1.plot(T_series, color='blue', label='RECD Time (T)')
ax1.set_xlabel('Calendar Time (Steps)')
ax1.set_ylabel('Accumulated Time T', color='blue')

ax2 = ax1.twinx()
ax2.plot(depths, color='orange', alpha=0.3, label='Renormalization Depth (k)')
ax2.set_ylabel('k', color='orange')

plt.title("Generation of Discrete Extramental Time")
plt.show()

## 4. Three-Layer Ontology

We apply the detectors to find relational windows (Joint Episodes) and structural reorganization (Capa 3).

In [ ]:
# Capa 1: Local Intensification
hp_z, core_hyper = st.hyper_persistence(taus_global)
lam, tt = st.rolling_rqa(taus_global)
M_series = st.critical_mass_metric(hp_z, lam, tt)

# Capa 2: Relational Coherence (A_antis)
A_series = st.compute_antisynchronization(taus_per_module)
episodes = st.extract_joint_episodes(A_series, M_series, theta_A=0.05, D_min=10)

print(f"Detected {len(episodes)} Joint Episodes (enabling windows).")

# Capa 3: Ontological Ascent
t_frob, max_dist = st.detect_reorganization_frob(taus_per_module)
t_ks, max_ks = st.detect_reorganization_ks(dtk_series)
t_star = st.consensus_transition(t_frob, t_ks)

print(f"Capa 3: Global reorganization detected at step t* = {t_star} (KS = {max_ks:.3f})")

## 5. Fractal Dimension Validation

Theoretically, the support of $T$ should approach $D \approx 1.98$. We estimate it using the Higuchi method.

In [ ]:
D = st.estimate_higuchi_dimension(T_series, k_max=20)
print(f"Estimated Fractal Dimension of T: {D:.3f} (Expected theoretical range: 1.96 - 2.01)")